# CGW S/N interactive explorer

This notebook is a deliberately inspectable version of the population pipeline. It lets you:

1. load the project pulsars and noise model;
2. generate a binary population, including its discrete background/strain summary;
3. optionally generate an S/N-consistent population with the project generator;
4. inject the population into the live libstempo pulsars;
5. rebuild the Enterprise PTA and calculate optimal CGW S/N; and
6. clone individual binaries, perturb their sky positions, and compare S/N and per-pulsar contributions.

The notebook keeps the important objects in local memory. Run cells in order, but inspect or modify any object between cells. Expensive cells are controlled by the switches in the configuration cell.

**Important:** the S/N calculation and the injection are separate operations. Injection mutates the pulsar objects; the optimal CGW S/N calculation evaluates the binary waveform against the PTA noise model. The notebook shows both paths explicitly.


In [ ]:
# Configuration: edit this cell before running the workflow.
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").exists():
    PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else PROJECT_ROOT

POPULATION_CONFIG = "test"       # test / optimistic / realistic / pessimistic
N_BINARIES = 200                  # keep small while debugging
RANDOM_SEED = 12345
PULSAR_SCENARIO = "baseline"
MIN_BASELINE_YEARS = 0.0

# Expensive operations are opt-in.
RUN_INJECTION = True
RUN_OPTIMAL_SNR = True
RUN_SNR_CONSISTENT = False
N_CONSISTENT_SIMS = 1
TARGET_POPULATION_SNR = 3.2
SNR_RANGE = (3.0, 3.4)

# Select a small set for rapid debugging, or use None for every loaded pulsar.
DEBUG_PULSAR_NAMES = None

# CGW diagnostic thresholds passed to the project implementation.
DEBUG_CGW = True
DEBUG_DENOMINATOR_THRESHOLD = 1e-3
DEBUG_RHO_SQ_THRESHOLD = 1.0


In [ ]:
# Imports and project paths
import copy
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from types import SimpleNamespace
from IPython.display import display

sys.path.insert(0, str(PROJECT_ROOT))

import config
import SMBHB_pop_synth as smbhb_module
from config import generate_population
from data_loader import (
    load_pulsars, filter_pulsars_15yr, get_clean_pulsars_and_tspan,
    parse_pulsar_parameters, restore_original_residuals,
)
from pta_builder import build_pta_and_params
from signal_injection import inject_population_nufft
from CGW_SNR import compute_cgw_snr_optimal_population_fast
from consistent_pop_synth import (
    generate_snr_consistent_populations_distance_scaling,
)
from enterprise.pulsar import Pulsar as EnterprisePulsar

print(f"Project root: {PROJECT_ROOT}")
print(f"Noise file:    {config.NOISEFILE}")


## 1. Load pulsars, noise parameters, and the PTA time span

`psrs` are the live libstempo objects. `original_stoas` and `original_residuals` are snapshots used to restore them after an injection. Do not overwrite these snapshots if you want to repeat controlled experiments in the same kernel.


In [ ]:
# Load and optionally restrict the live pulsar set.
psrs = load_pulsars(verbose=True, scenario=PULSAR_SCENARIO)
psrs, _, Tspan_seconds = filter_pulsars_15yr(
    psrs, min_baseline_years=MIN_BASELINE_YEARS, verbose=True
)

if DEBUG_PULSAR_NAMES is not None:
    wanted = set(DEBUG_PULSAR_NAMES)
    psrs = [psr for psr in psrs if psr.name in wanted]

if not psrs:
    raise RuntimeError("No pulsars remain. Check the data paths or DEBUG_PULSAR_NAMES.")

# Snapshot the state before any injection.
original_stoas = {psr.name: np.array(psr.stoas, copy=True) for psr in psrs}
original_residuals = {psr.name: np.array(psr.residuals, copy=True) for psr in psrs}
raw_noise_params = json.loads(Path(config.NOISEFILE).read_text())
parsed_noise_params = parse_pulsar_parameters(config.NOISEFILE)

print(f"Using {len(psrs)} pulsars")
print(f"Tspan = {Tspan_seconds / (365.25 * 86400):.4f} yr")
print("First pulsars:", [psr.name for psr in psrs[:10]])


In [ ]:
# Compact pulsar geometry/noise table for inspection.
psr_rows = []
for psr in psrs:
    psr_rows.append({
        "name": psr.name,
        "n_toa": len(psr.toas),
        "ra_rad": float(psr._raj),
        "dec_rad": float(psr._decj),
        "ra_deg": np.degrees(float(psr._raj)),
        "dec_deg": np.degrees(float(psr._decj)),
        "toa_min_yr": float(np.min(psr.toas) / (365.25 * 86400)),
        "toa_max_yr": float(np.max(psr.toas) / (365.25 * 86400)),
    })
psr_table = pd.DataFrame(psr_rows).sort_values("name")
display(psr_table)


## 2. Generate a population and inspect its in-memory binary properties

`population` is the project `PopulationArrays` object. It contains the arrays used by the injection and S/N code, so changing an array entry changes the in-memory source. `strain_data` contains the discrete population background summary when `compute_strain=True`.


In [ ]:
population_config = dict(config.POPULATION_CONFIGS[POPULATION_CONFIG])
population_config["n_binaries"] = N_BINARIES

population, strain_data = generate_population(
    population_config,
    smbhb_module,
    T_obs_seconds=Tspan_seconds,
    n_binaries=N_BINARIES,
    compute_strain=True,
    seed=RANDOM_SEED,
)

print(f"Generated {len(population):,} binaries")
print("Approximate source-array memory:", f"{population.memory_gb():.4f} GB")
print("Available source fields:", [k for k in vars(population) if not k.startswith("amp_")])
if strain_data is not None:
    print("Background fields:", sorted(strain_data))


In [ ]:
# Inspect the discrete background represented by this population.
if strain_data is not None:
    background_table = pd.DataFrame({
        "frequency_hz": np.asarray(strain_data["bin_centres"], dtype=float),
        "h_c_total": np.asarray(strain_data["h_c_total"], dtype=float),
    })
    background_table = background_table.replace([np.inf, -np.inf], np.nan).dropna()
    display(background_table.head(20))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.loglog(background_table["frequency_hz"], background_table["h_c_total"], marker="o")
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel(r"Characteristic strain $h_c$")
    ax.set_title("Discrete SMBHB population background")
    ax.grid(True, which="both", alpha=0.25)
    plt.show()
else:
    print("No strain_data available; generate with compute_strain=True.")


In [ ]:
def population_table(pop, indices=None):
    """Return selected PopulationArrays entries as a readable DataFrame."""
    if indices is None:
        indices = np.arange(len(pop))
    indices = np.asarray(indices, dtype=int)
    fields = ("f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc",
              "ra", "dec", "psi", "iota", "phi0", "cgw_snr")
    return pd.DataFrame({field: np.asarray(getattr(pop, field))[indices] for field in fields},
                        index=indices)

def nearest_sources(pop, ra, dec, n=10):
    """Return source indices nearest to a sky position in great-circle distance."""
    cos_sep = (np.sin(pop.dec) * np.sin(dec)
               + np.cos(pop.dec) * np.cos(dec) * np.cos(pop.ra - ra))
    sep = np.arccos(np.clip(cos_sep, -1.0, 1.0))
    idx = np.argsort(sep)[:n]
    out = population_table(pop, idx).copy()
    out["separation_deg"] = np.degrees(sep[idx])
    return out.sort_values("separation_deg")

# Inspect the first sources and the sources nearest a chosen sky location.
display(population_table(population, np.arange(min(10, len(population)))))
source_index = int(np.argmax(np.asarray(population.h0)))
print("Highest-h0 source index:", source_index)
display(population_table(population, [source_index]))
display(nearest_sources(population, population.ra[source_index], population.dec[source_index], n=5))


## 3. Optional S/N-consistent population generation

This uses the project’s distance-scaling generator. It can resimulate pulsar noise and is substantially more expensive than the direct population generation above. The result is retained in `consistent_results`; each entry contains the generated population and S/N metadata.


In [ ]:
consistent_results = None
if RUN_SNR_CONSISTENT:
    consistent_results = generate_snr_consistent_populations_distance_scaling(
        config_template=population_config,
        smbhb_module=smbhb_module,
        psrs_clean=psrs,
        raw_noise_params=raw_noise_params,
        Tspan=Tspan_seconds,
        target_SNR=TARGET_POPULATION_SNR,
        SNR_range=SNR_RANGE,
        original_stoas=original_stoas,
        N_sims=N_CONSISTENT_SIMS,
        verbose=True,
        save_populations=False,
        profile=True,
        keep_amplitudes_in_result=False,
    )
    print("Consistent-population result keys:", consistent_results.keys())
    print("Successful populations:", len(consistent_results.get("populations", [])))
else:
    print("Skipped: set RUN_SNR_CONSISTENT=True to run the expensive consistency generator.")


## 4. Inject the population into the live pulsars

This mutates `psr.stoas` through the same NUFFT path used by the pipeline. The pre-injection state can always be restored with the next cell. For a source-by-source residual experiment, make a one-source population or use the `inject_population_nufft` call below with a selected slice.


In [ ]:
if RUN_INJECTION:
    # Restore first so re-running this cell is deterministic.
    for psr in psrs:
        psr.stoas[:] = original_stoas[psr.name]
        psr.residuals[:] = original_residuals[psr.name]

    inject_population_nufft(
        psrs, population, verbose=True, eps=1e-6,
        cache_precomputed_amplitudes=True,
        track_contributors=True, top_k_global=20,
    )
    injected_stoas = {psr.name: np.array(psr.stoas, copy=True) for psr in psrs}
    print("Injection complete")
    if hasattr(population, "contributor_summary"):
        display(pd.DataFrame(population.contributor_summary))
else:
    print("Skipped injection: set RUN_INJECTION=True to mutate the live pulsars.")

# Restore before constructing a fresh Enterprise snapshot for the optimal-S/N path.
for psr in psrs:
    psr.stoas[:] = original_stoas[psr.name]
    psr.residuals[:] = original_residuals[psr.name]


## 5. Build the Enterprise PTA and calculate CGW S/N

The Enterprise snapshot is separate from the mutable libstempo objects. `enterprise_psrs`, `pta`, and `pta_params` are retained so the PTA basis and noise matrices can be inspected. Set `DEBUG_CGW=False` for a quiet run, or leave it enabled to print the detailed geometry/residual/inner-product trace for sources crossing the configured thresholds.


In [ ]:
enterprise_psrs = [EnterprisePulsar(psr, ephem="DE440", backend="tempo2") for psr in psrs]
pta, pta_model, pta_params = build_pta_and_params(
    psrs=enterprise_psrs,
    noise_params=raw_noise_params,
    Tspan=Tspan_seconds,
    include_GW=True, include_RN=True, include_WN=True,
)
print(f"PTA pulsars: {len(pta.pulsars)}")
print(f"PTA parameters: {len(pta.params)}")

if RUN_OPTIMAL_SNR:
    snrs, breakdowns = compute_cgw_snr_optimal_population_fast(
        psrs=enterprise_psrs,
        pta=pta,
        population=population,
        raw_noise_params=raw_noise_params,
        parsed_noise_params=parsed_noise_params,
        Tspan=Tspan_seconds,
        profile=True,
        return_breakdown=True,
        debug=DEBUG_CGW,
        debug_denominator_threshold=DEBUG_DENOMINATOR_THRESHOLD,
        debug_rho_sq_threshold=DEBUG_RHO_SQ_THRESHOLD,
    )
    population.cgw_snr[:] = np.asarray(snrs, dtype=population.cgw_snr.dtype)
    print("S/N range:", float(np.min(snrs)), float(np.max(snrs)))
else:
    snrs, breakdowns = None, None
    print("Skipped optimal S/N calculation.")


In [ ]:
# Rank the sources by recalculated CGW S/N and expose per-pulsar contributions.
if snrs is not None:
    ranked = np.argsort(np.asarray(snrs))[::-1]
    snr_table = population_table(population, ranked[:20]).copy()
    snr_table["recomputed_cgw_snr"] = np.asarray(snrs)[ranked[:20]]
    display(snr_table[["recomputed_cgw_snr", "f", "Mc", "h0", "ra", "dec", "D_comov"]])

    selected_index = int(ranked[0])
    contribution_table = (
        pd.Series(breakdowns[selected_index], name="rho_sq")
        .sort_values(ascending=False)
        .to_frame()
    )
    contribution_table["fraction"] = contribution_table["rho_sq"] / contribution_table["rho_sq"].sum()
    print(f"Selected source index: {selected_index}")
    display(contribution_table.head(20))


## 6. Controlled binary clones and sky-position perturbations

The helper below turns one source row into a scalar object accepted by the CGW S/N implementation. It preserves every intrinsic/orientation parameter and changes only the requested fields. This is the main debugging surface for testing whether a nearby pulsar, a coordinate rounding difference, or another parameter drives an S/N change.


In [ ]:
SOURCE_FIELDS = ("f", "Mc", "Mtot", "D_comov", "z", "h0", "ecc",
                 "ra", "dec", "psi", "iota", "phi0", "cgw_snr")

def binary_from_population(pop, index):
    """Copy one source into an independent scalar binary object."""
    index = int(index)
    values = {field: float(np.asarray(getattr(pop, field))[index]) for field in SOURCE_FIELDS}
    return SimpleNamespace(**values)

def clone_binary(binary, **changes):
    """Clone a binary and change only explicitly supplied attributes."""
    values = {field: float(getattr(binary, field)) for field in SOURCE_FIELDS}
    unknown = set(changes) - set(values)
    if unknown:
        raise KeyError(f"Unknown binary fields: {sorted(unknown)}")
    values.update({key: float(value) for key, value in changes.items()})
    return SimpleNamespace(**values)

def evaluate_binaries(binaries, debug=False, return_breakdown=True):
    """Evaluate a small list of scalar binaries with the current PTA."""
    return compute_cgw_snr_optimal_population_fast(
        psrs=enterprise_psrs, pta=pta, population=list(binaries),
        raw_noise_params=raw_noise_params, parsed_noise_params=parsed_noise_params,
        Tspan=Tspan_seconds, profile=True, return_breakdown=return_breakdown,
        debug=debug, debug_denominator_threshold=DEBUG_DENOMINATOR_THRESHOLD,
        debug_rho_sq_threshold=DEBUG_RHO_SQ_THRESHOLD,
    )

reference_binary = binary_from_population(population, selected_index if snrs is not None else source_index)
print("Reference binary:")
display(pd.DataFrame([vars(reference_binary)]))


In [ ]:
# Example: identical binary, with controlled RA/Dec perturbations.
# Change perturbations in radians; 1e-6 rad is about 0.000057 degrees.
perturbations = [
    ("reference", 0.0, 0.0),
    ("RA + 1e-6 rad", 1e-6, 0.0),
    ("Dec + 1e-6 rad", 0.0, 1e-6),
    ("RA + 1e-4 rad", 1e-4, 0.0),
]

comparison_binaries = [
    clone_binary(reference_binary, ra=reference_binary.ra + dra, dec=reference_binary.dec + ddec)
    for _, dra, ddec in perturbations
]
comparison_snrs, comparison_breakdowns = evaluate_binaries(comparison_binaries, debug=False)
comparison_rows = []
for (label, dra, ddec), value, breakdown in zip(perturbations, comparison_snrs, comparison_breakdowns):
    comparison_rows.append({
        "case": label, "delta_ra_rad": dra, "delta_dec_rad": ddec,
        "delta_ra_arcsec": np.degrees(dra) * 3600,
        "delta_dec_arcsec": np.degrees(ddec) * 3600,
        "cgw_snr": float(value),
        "rho_sq": float(value) ** 2,
        "largest_pulsar": max(breakdown, key=breakdown.get),
        "largest_rho_sq": max(breakdown.values()),
    })
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)


In [ ]:
# Compare the reference source against one deliberately nearby sky position with full tracing.
# This prints the source, pulsar geometry, antenna denominator, waveform scale,
# residual statistics, inner product, and cumulative rho^2 for triggered pulsars.
nearby_binary = clone_binary(reference_binary, dec=reference_binary.dec + 1e-5)
nearby_snr, nearby_breakdown = evaluate_binaries([reference_binary, nearby_binary], debug=True)
print("Reference / nearby SNR:", nearby_snr)
print("SNR ratio:", float(nearby_snr[1] / nearby_snr[0]) if nearby_snr[0] else np.nan)


## 7. Restore the live pulsars

Run this cell before leaving the notebook or before starting another experiment. It returns the libstempo objects to the exact pre-injection TOAs and residuals captured near the beginning.


In [ ]:
for psr in psrs:
    psr.stoas[:] = original_stoas[psr.name]
    psr.residuals[:] = original_residuals[psr.name]
print("Restored live pulsars to their pre-injection state.")
